# Phase 4 - Court judgment prediction with Arch 4 - DeepSeek-style MoE decoder

Starts from the Phase-1 model (12-layer MoE decoder (222.6M total, ~81M active): MLA attention, 32 routed + 1 shared expert per layer.) and predicts the outcome (0 / 1) of a case. Metric: macro-F1.

**Reading a long case.** Judgments average ~20K characters (up to 2.5M). Each case becomes four 512-token chunks: the
opening of the judgment plus its **last three chunks** - in ILDC (Malik et al. 2021, the source of this data) the
closing part, right before the removed verdict, carried the most signal. The Phase-1 backbone encodes every chunk
(causal mask lifted for the decoder-only models), chunks are mean-pooled, tagged with the section they came from, and
merged by **learned attention pooling** into one document vector - a hierarchical chunk -> document representation.

**Class balance.** The training set is 59% class 0 but dev and test are 50/50, so the loss is class-balanced and the
decision threshold is tuned on the dev set (it is the same distribution as the test set).

**Experiments** (each is its own submission; all from the Phase-4 list)

| Experiment | What trains |
|---|---|
| `full` | every weight, layer-wise LR decay - strong reference and the distillation teacher |
| `lorma` | **LoRMA** (compulsory): `(alpha/r BA + I) W x` on the attention projections (Bihany, Patel & Modi 2025) |
| `bitfit` | only bias terms (Ben Zaken et al. 2022); bias-free layers get a zero-initialised bias |
| `adapter` | Houlsby bottleneck adapters after the attention and feed-forward outputs (Houlsby et al. 2019) |
| `unfreeze` | gradual unfreezing: the head first, then one more backbone layer at a time from the top (Howard & Ruder 2018) |
| `kd` | distillation: the fine-tuned `full` model teaches a half-depth student via its logits and its document vector (Hinton et al. 2015) |

There are 37K training cases - more than a Kaggle session can read in full - so every experiment trains for a fixed
**time budget** (`TRAIN_MINUTES`) and the learning-rate schedule is fitted to that budget.

**Note:** the public `test.json` also contains a `label` field. It is never read here - only `id` and `text`.

**Kaggle settings:** GPU T4 x2, Internet on, secret `HF` holding your Hugging Face token. Then Run All.

## 1. Setup

In [ ]:
import os, sys, json, glob, shutil, time, gc
import numpy as np
import torch
from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

WORK_DIR = "/kaggle/working/phase4_arch4"
os.makedirs(WORK_DIR, exist_ok=True)
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 2. Configuration
Everything you might want to change lives here.

In [ ]:
%%writefile /kaggle/working/phase4_arch4/config.py
"""Every knob of this Phase-4 (court judgment prediction) notebook."""
import os
import json

ARCH = "arch4"
WORK_DIR = f"/kaggle/working/phase4_{ARCH}"
DATA_DIR = os.path.join(WORK_DIR, "data")
CACHE_DIR = os.path.join(WORK_DIR, "phase1")        # Phase-1 files downloaded from Hugging Face land here
OUT_DIR = os.path.join(WORK_DIR, "runs")             # one folder per experiment

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
RAW = {s: os.path.join(DATA_DIR, "Phase_4_CJPE", "Test" if s == "test" else "Train", f"{s}.json")
       for s in ("train", "dev", "test")}

# ------------------------------------------------------------------ the Phase-1 model
# Looked up in this order: an attached Kaggle dataset whose path mentions "arch4", then these HF repos.
# The 600M-token run (v2) is preferred; the earlier 100M-token run is the fallback.
PHASE1_REPOS = ["D-240357/legal-arch4-moe-v2", "D-240357/legal-arch4-moe"]
WEIGHT_FILES = ["model_final.pt", "model_best.pt"]
TOKENIZER_FILES = ["tokenizer.json", "data/tokenizer.json"]
TOKENIZER_REPOS = PHASE1_REPOS + ["D-240357/legal-arch3-decoder"]
KAGGLE_HINT = "arch4"

CJPE_REPO = "D-240357/legal-cjpe-arch4"         # fine-tuned checkpoints, results and submission zips go here

# ------------------------------------------------------------------ experiments
# Each one is a separate submission. "full" is the strong reference and the distillation teacher;
# "kd" waits for it to finish.
METHODS = ["full", "lorma", "bitfit", "adapter", "unfreeze", "kd"]
KD_TEACHER = "full"
TRAIN_MINUTES = 35             # wall-clock training budget per experiment (the LR schedule ends exactly on it)
MAX_EPOCHS = 2                 # upper bound if the GPU is fast enough to get through the data

# ------------------------------------------------------------------ how a case is read
CHUNK_TOKENS = 512
HEAD_CHUNKS = 1                # opening of the judgment (parties, facts)
TAIL_CHUNKS = 3                # its last part, right before the removed verdict - the most telling section in ILDC
CHARS_PER_TOKEN = 8            # only this much text is tokenized from each end (keeps the preprocessing fast)
BIDIRECTIONAL = True            # lift the causal mask while fine-tuning (Dukic & Snajder 2024; LLM2Vec)
POOL_DIM = 256                 # attention pooling across chunks
HEAD_DROPOUT = 0.1

# ------------------------------------------------------------------ optimisation
MICRO_DOCS = 2                 # cases per forward pass (each case = up to 4 chunks)
GRAD_ACCUM = 8
EVAL_BATCH = 8
WARMUP_STEPS = 50
MIN_LR_RATIO = 0.1
WEIGHT_DECAY = 0.01
HEAD_LR = 1e-3
FULL_LR = 5e-5                 # top layer; each layer below gets LLRD times the one above it
LLRD = 0.9
ADAPTER_LR = 5e-4
BITFIT_LR = 1e-3
ADAPTER_RANK = 8
LORMA_ALPHA = 8
ADAPTER_DROPOUT = 0.05
ADAPTER_TARGETS = ["q_down", "q_up", "kv_down", "kv_up", "out"]       # LoRMA: attention projections
ADAPTER_SITES = ["out", "down"]           # Houlsby adapters: after the attention output and the feed-forward output
ADAPTER_BOTTLENECK = 64
UNFREEZE_BY = 0.5              # gradual unfreezing: every layer is trainable after half the budget

KD_TEMPERATURE = 2.0
KD_ALPHA = 0.5                 # weight of the gold-label loss; the rest goes to matching the teacher
KD_HIDDEN_WEIGHT = 0.5         # also align the student's document vector with the teacher's (cosine)
STUDENT_LAYERS = [0, 2, 4, 6, 8, 10]

# ------------------------------------------------------------------ evaluation
EVAL_EVERY_STEPS = 150         # quick check on DEV_SUBSET cases; the best checkpoint is kept
DEV_SUBSET = 600
DEV_FINAL_LIMIT = None         # the final threshold is tuned on the whole dev set (3,505 cases)
TRAIN_DOC_LIMIT = None         # set to e.g. 400 (and TRAIN_MINUTES = 3) for a quick smoke test
PREP_WORKERS = max(1, os.cpu_count() or 2)

SEED = 13

_override = os.environ.get("P4_CFG_OVERRIDE")
if _override:
    globals().update(json.loads(_override))

## 3. Code
The Phase-1 tokenizer and model (unchanged, so the trained weights load), plus the judgment-prediction pieces.

In [ ]:
%%writefile /kaggle/working/phase4_arch4/tokenizer.py
"""
Byte-level BPE tokenizer, written from scratch (stdlib only for the tokenizer logic).

How it works, in short:
  1. A regex chops text into "pre-tokens" (a word with its leading space, a run of digits,
     a run of punctuation, ...). Merges never cross these boundaries.
  2. Every pre-token is turned into raw UTF-8 bytes (ids 0..255), so *any* text can be
     encoded - there is no <unk> and no lowercasing.
  3. Training repeatedly merges the most frequent adjacent id pair into a new id.
     We train on the table of *unique* pre-tokens with their counts, which is what keeps
     it fast in pure Python.
  4. Encoding replays the learned merges (lowest rank first) per pre-token, and caches the
     result, so repeated words cost a single dict lookup.
"""
import re
import json
import heapq
from collections import Counter, defaultdict

# Close to the GPT-2 splitter, but built on plain `re` (no \p{L}), so it needs no extra package.
PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+",
    re.UNICODE,
)

EOS_TOKEN = "<|endoftext|>"
PAD_TOKEN = "<|pad|>"


class ByteBPE:
    def __init__(self, merges=None, vocab_size=32000):
        self.vocab_size = vocab_size
        self.merges = [tuple(m) for m in (merges or [])]      # merge i creates id 256 + i
        self.eos_id = vocab_size - 2
        self.pad_id = vocab_size - 1
        self._cache = {}
        self._rebuild_tables()

    # ------------------------------------------------------------- internals
    def _rebuild_tables(self):
        self.ranks = {pair: (i, 256 + i) for i, pair in enumerate(self.merges)}
        vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            vocab.append(vocab[a] + vocab[b])
        self.vocab = vocab
        self._cache.clear()

    # -------------------------------------------------------------- training
    @classmethod
    def train(cls, texts, vocab_size=32000, log_every=2000):
        """Learn merges from an iterable of strings."""
        n_merges = vocab_size - 256 - 2          # leave room for the two special tokens
        word_counts = Counter()
        for text in texts:
            word_counts.update(PRETOKEN_RE.findall(text))
        print(f"  unique pre-tokens: {len(word_counts):,}")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_count = defaultdict(int)
        where = defaultdict(set)                 # pair -> indices of words that (may) contain it
        for wi, w in enumerate(words):
            f = freqs[wi]
            for pair in zip(w, w[1:]):
                pair_count[pair] += f
                where[pair].add(wi)

        # Max-heap with lazy invalidation: stale entries are fixed up when they surface.
        heap = [(-c, p) for p, c in pair_count.items()]
        heapq.heapify(heap)

        merges = []
        vocab = [bytes([i]) for i in range(256)]
        for m in range(n_merges):
            best = None
            while heap:
                neg, pair = heapq.heappop(heap)
                real = pair_count.get(pair, 0)
                if real <= 0:
                    continue
                if real != -neg:
                    heapq.heappush(heap, (-real, pair))
                    continue
                best = pair
                break
            if best is None:
                print(f"  ran out of pairs after {m} merges")
                break

            new_id = 256 + m
            merges.append(best)
            a, b = best
            vocab.append(vocab[a] + vocab[b])
            touched = {}
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                # find the occurrences in this word (left to right, non-overlapping)
                if len(w) < 2:
                    continue
                i, out, changed = 0, [], False
                while i < len(w):
                    if i < len(w) - 1 and w[i] == a and w[i + 1] == b:
                        out.append(new_id)
                        i += 2
                        changed = True
                    else:
                        out.append(w[i])
                        i += 1
                if not changed:
                    continue                      # stale entry in `where`
                for pair in zip(w, w[1:]):        # take the old word's pairs away ...
                    pair_count[pair] -= f
                for pair in zip(out, out[1:]):    # ... and add the new word's pairs
                    pair_count[pair] += f
                    where[pair].add(wi)
                    touched[pair] = pair_count[pair]
                words[wi] = out
            for pair, c in touched.items():
                if c > 0:
                    heapq.heappush(heap, (-c, pair))
            pair_count.pop(best, None)

            if (m + 1) % log_every == 0:
                print(f"  merge {m + 1:,}/{n_merges:,}  last = {vocab[-1]!r}")

        return cls(merges, vocab_size)

    # -------------------------------------------------------------- encoding
    def _encode_word(self, word):
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) > 1:
            best_rank, best_pair, best_new = None, None, None
            for pair in zip(ids, ids[1:]):
                hit = ranks.get(pair)
                if hit is not None and (best_rank is None or hit[0] < best_rank):
                    best_rank, best_pair, best_new = hit[0], pair, hit[1]
            if best_pair is None:
                break
            a, b = best_pair
            out, i = [], 0
            while i < len(ids):
                if i < len(ids) - 1 and ids[i] == a and ids[i + 1] == b:
                    out.append(best_new)
                    i += 2
                else:
                    out.append(ids[i])
                    i += 1
            ids = out
        return ids

    def encode(self, text):
        cache = self._cache
        if len(cache) > 1_000_000:               # keep RAM bounded on huge corpora
            cache.clear()
        out = []
        for word in PRETOKEN_RE.findall(text):
            ids = cache.get(word)
            if ids is None:
                ids = cache[word] = self._encode_word(word)
            out.extend(ids)
        return out

    def decode(self, ids):
        specials = (self.eos_id, self.pad_id)
        data = b"".join(self.vocab[i] for i in ids if i < len(self.vocab) and i not in specials)
        return data.decode("utf-8", errors="replace")

    # ----------------------------------------------------------- persistence
    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"vocab_size": self.vocab_size, "merges": self.merges,
                       "eos": EOS_TOKEN, "pad": PAD_TOKEN}, f)

    @classmethod
    def load(cls, path):
        with open(path, "r", encoding="utf-8") as f:
            blob = json.load(f)
        return cls(blob["merges"], blob["vocab_size"])


# ---------------------------------------------------------------------------------------------------
# Phase 2 addition: encode with character offsets, so predicted tokens can be mapped back to exact spans.
# ---------------------------------------------------------------------------------------------------
# Same splitter as above, except a run of punctuation is cut into single characters. Gold entities often end
# in the middle of runs like "Ors.[1]" or "Cr.P.C.,"; with this split 99.3% of gold spans start and end on a
# token boundary (98.1% without it). Every piece is still encoded with the Phase-1 merges.
NER_PRETOKEN_RE = re.compile(
    r"'(?:s|t|re|ve|m|ll|d)| ?[^\W\d_]+| ?\d+| ?[^\s\w]|\s+(?!\S)|\s+",
    re.UNICODE,
)


class NerTokenizer(ByteBPE):
    split_punct = True

    def encode_with_offsets(self, text):
        """Returns ids and, per token, the [start, end) of its non-whitespace characters in `text`."""
        splitter = NER_PRETOKEN_RE if self.split_punct else PRETOKEN_RE
        ids, starts, ends = [], [], []
        for m in splitter.finditer(text):
            word, base = m.group(), m.start()
            toks = self._cache.get(word)
            if toks is None:
                toks = self._cache[word] = self._encode_word(word)
            # byte position -> character position inside this piece (pieces can contain multi-byte characters)
            char_at_byte = []
            for ci, ch in enumerate(word):
                char_at_byte += [ci] * len(ch.encode("utf-8"))
            b = 0
            for t in toks:
                n = len(self.vocab[t])
                cs, ce = char_at_byte[b], char_at_byte[b + n - 1] + 1
                b += n
                piece = word[cs:ce]
                s = base + cs + (len(piece) - len(piece.lstrip()))
                e = base + ce - (len(piece) - len(piece.rstrip()))
                if s >= e:
                    s = e = base + ce
                ids.append(t)
                starts.append(s)
                ends.append(e)
        return ids, starts, ends

In [ ]:
%%writefile /kaggle/working/phase4_arch4/phase1_model.py
"""
Phase-1 model code, unchanged except for the small switches marked "Phase 2", so the saved weights load as-is.
"""
"""
Arch 4: a small DeepSeek-V3-flavoured mixture-of-experts decoder, written from scratch.

Ideas borrowed from the DeepSeek papers (scaled down to ~217M total / ~80M active parameters):

  * MLA - Multi-head Latent Attention. Keys/values are squeezed through a small latent vector,
    so the KV cache holds `kv_lora + qk_rope` numbers per token instead of full K and V.
    RoPE lives on a small separate slice of each head ("decoupled RoPE") so the latent stays
    position-free.
  * DeepSeekMoE - many small routed experts plus one shared expert that every token uses.
  * Auxiliary-loss-free load balancing (V3) - a per-expert bias is added to the routing scores
    *only to pick the top-k*; after every optimizer step it is nudged up for under-used experts
    and down for over-used ones. Gate weights come from the un-biased sigmoid scores. A tiny
    sequence-wise balance loss is kept only as a safety net.
  * Multi-token prediction (MTP) - during training an extra light module predicts the token two
    steps ahead. It shares the embedding / output head and is simply unused at inference.

Our own adaptation: expert dispatch uses fixed-size capacity buffers and one batched matmul over
all experts. Every tensor shape is static, so torch.compile can fuse it and there is no Python
loop over experts. In eval / generation the capacity is raised so that no token is ever dropped.
"""
import math
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist


@dataclass
class ModelConfig:
    vocab_size: int = 32000
    n_layer: int = 12
    n_head: int = 12
    d_model: int = 768
    max_seq_len: int = 2048
    rope_base: float = 10000.0
    norm_eps: float = 1e-5
    # MLA
    q_lora: int = 384
    kv_lora: int = 192
    qk_nope: int = 48
    qk_rope: int = 16
    v_dim: int = 64
    # MoE
    n_dense_layers: int = 1
    d_ff_dense: int = 2048
    n_routed: int = 32
    top_k: int = 4
    expert_ff: int = 192
    shared_ff: int = 512
    capacity_factor: float = 1.25
    # multi-token prediction
    mtp: bool = True
    mtp_ff: int = 1024


# ---------------------------------------------------------------------------
# small building blocks
# ---------------------------------------------------------------------------
class RMSNorm(nn.Module):
    def __init__(self, dim, eps):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x32 = x.float()
        x32 = x32 * torch.rsqrt(x32.pow(2).mean(-1, keepdim=True) + self.eps)
        return (x32 * self.weight).to(x.dtype)


def build_rope_cache(dim, max_len, base):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1).to(x.dtype)


class SwiGLU(nn.Module):
    def __init__(self, d_model, hidden):
        super().__init__()
        self.gate_up = nn.Linear(d_model, 2 * hidden, bias=False)
        self.down = nn.Linear(hidden, d_model, bias=False)

    def forward(self, x):
        gate, up = self.gate_up(x).chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)


# ---------------------------------------------------------------------------
# Multi-head Latent Attention
# ---------------------------------------------------------------------------
class MLA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.H = cfg.n_head
        self.nope, self.rope, self.vd = cfg.qk_nope, cfg.qk_rope, cfg.v_dim
        self.kv_lora = cfg.kv_lora

        self.q_down = nn.Linear(cfg.d_model, cfg.q_lora, bias=False)
        self.q_norm = RMSNorm(cfg.q_lora, cfg.norm_eps)
        self.q_up = nn.Linear(cfg.q_lora, self.H * (self.nope + self.rope), bias=False)

        # one projection yields the compressed KV latent AND the (shared) rotary key slice
        self.kv_down = nn.Linear(cfg.d_model, cfg.kv_lora + self.rope, bias=False)
        self.kv_norm = RMSNorm(cfg.kv_lora, cfg.norm_eps)
        self.kv_up = nn.Linear(cfg.kv_lora, self.H * (self.nope + self.vd), bias=False)

        self.out = nn.Linear(self.H * self.vd, cfg.d_model, bias=False)
        self.causal = True                         # Phase 2 turns this off to let tokens look right

    def forward(self, x, cos, sin, past=None, key_mask=None):
        B, T, _ = x.shape
        H = self.H

        q = self.q_up(self.q_norm(self.q_down(x))).view(B, T, H, self.nope + self.rope).transpose(1, 2)
        q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
        q_rope = apply_rope(q_rope, cos, sin)

        kv = self.kv_down(x)
        latent, k_rope = kv.split([self.kv_lora, self.rope], dim=-1)
        latent = self.kv_norm(latent)                                   # (B, T, kv_lora)
        k_rope = apply_rope(k_rope.unsqueeze(1), cos, sin)              # (B, 1, T, rope) shared by all heads

        if past is not None:                                            # cache = latent + rope key only
            latent = torch.cat([past[0], latent], dim=1)
            k_rope = torch.cat([past[1], k_rope], dim=2)
        present = (latent, k_rope)

        S = latent.size(1)
        kv_full = self.kv_up(latent).view(B, S, H, self.nope + self.vd).transpose(1, 2)
        k_nope, v = kv_full.split([self.nope, self.vd], dim=-1)

        q = torch.cat([q_nope, q_rope], dim=-1)
        k = torch.cat([k_nope, k_rope.expand(-1, H, -1, -1)], dim=-1)

        if key_mask is None:
            o = F.scaled_dot_product_attention(q, k, v, is_causal=self.causal)
        else:
            o = F.scaled_dot_product_attention(q, k, v, attn_mask=key_mask)
        o = o.transpose(1, 2).reshape(B, T, H * self.vd)
        return self.out(o), present


# ---------------------------------------------------------------------------
# DeepSeekMoE layer with auxiliary-loss-free balancing and static-shape dispatch
# ---------------------------------------------------------------------------
@torch.compiler.disable
def _slot_positions(eid, E):
    """
    For every (token, expert) pair: its running position inside that expert's buffer, plus the
    one-hot assignment matrix. This is integer bookkeeping on ~33K elements, so it runs eagerly
    on purpose: comparing an index tensor with arange(E) trips an Inductor/Triton codegen bug on
    some torch builds (seen on Kaggle). Everything heavy around it stays compiled.
    """
    onehot = (eid.unsqueeze(1) == torch.arange(E, device=eid.device).unsqueeze(0)).to(torch.int32)
    pos = onehot.cumsum(0).gather(1, eid.unsqueeze(1)).squeeze(1) - 1
    return onehot, pos


class MoE(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.E, self.K = cfg.n_routed, cfg.top_k
        self.cap_factor = cfg.capacity_factor
        self.dropless = False                      # Phase 2 fine-tuning routes every token
        D, h = cfg.d_model, cfg.expert_ff

        self.router = nn.Linear(D, self.E, bias=False)
        self.w_gate_up = nn.Parameter(torch.empty(self.E, D, 2 * h))    # all experts stacked -> one bmm
        self.w_down = nn.Parameter(torch.empty(self.E, h, D))
        self.shared = SwiGLU(D, cfg.shared_ff)

        nn.init.normal_(self.w_gate_up, std=0.02)
        nn.init.normal_(self.w_down, std=0.02)

        self.register_buffer("bias", torch.zeros(self.E))               # saved: it is learned state
        self.register_buffer("load", torch.zeros(self.E), persistent=False)
        self.register_buffer("dropped", torch.zeros(()), persistent=False)

    def forward(self, x):
        B, T, D = x.shape
        N, E, K = B * T, self.E, self.K
        flat = x.reshape(N, D)

        # --- routing: sigmoid affinities, biased top-k, normalised gate weights
        scores = torch.sigmoid(F.linear(flat.float(), self.router.weight.float()))      # (N, E)
        _, idx = torch.topk(scores.detach() + self.bias, K, dim=-1)                     # bias steers selection only
        gate = scores.gather(1, idx)
        gate = gate / (gate.sum(-1, keepdim=True) + 1e-9)

        # --- slot assignment: position of every (token, expert) pair inside its expert's buffer
        eid = idx.reshape(-1)                                                           # token-major, (N*K,)
        onehot, pos = _slot_positions(eid, E)

        if self.training and not self.dropless:
            cap = int(math.ceil(N * K / E * self.cap_factor / 8) * 8)
        else:
            cap = N                                                                     # drop-free for eval / generation
        keep = pos < cap
        slot = torch.where(keep, eid * cap + pos, torch.full_like(eid, E * cap))        # overflow -> spare row

        # --- dispatch -> one batched matmul over all experts -> combine
        dt = torch.get_autocast_dtype("cuda") if (torch.is_autocast_enabled() and x.is_cuda) else x.dtype
        xs = flat.to(dt).repeat_interleave(K, dim=0)
        buf = xs.new_zeros(E * cap + 1, D).index_put((slot,), xs)
        hid = buf[: E * cap].view(E, cap, D)
        g, u = torch.bmm(hid, self.w_gate_up.to(dt)).chunk(2, dim=-1)
        y = torch.bmm(F.silu(g) * u, self.w_down.to(dt))                                # (E, cap, D)
        y = torch.cat([y.reshape(E * cap, D), y.new_zeros(1, D)], dim=0)
        routed = (y[slot].view(N, K, D) * gate.to(y.dtype).unsqueeze(-1)).sum(1)

        out = (routed.to(x.dtype) + self.shared(flat)).view(B, T, D)

        if not self.training:
            return out, x.new_zeros((), dtype=torch.float32)

        # --- bookkeeping for balancing (no gradient) and the small sequence-wise safety loss
        with torch.no_grad():
            self.load += onehot.sum(0).float()
            self.dropped += (~keep).sum().float()
        p = (scores / (scores.sum(-1, keepdim=True) + 1e-9)).view(B, T, E).mean(1)      # mean affinity per sequence
        f = E * onehot.view(B, T * K, E).float().mean(1)                                # share of assignments per expert
        aux = (f * p).sum(-1).mean()
        return out, aux


# ---------------------------------------------------------------------------
# transformer block, MTP head, full model
# ---------------------------------------------------------------------------
class Block(nn.Module):
    def __init__(self, cfg, use_moe):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLA(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.use_moe = use_moe
        self.ffn = MoE(cfg) if use_moe else SwiGLU(cfg.d_model, cfg.d_ff_dense)

    def forward(self, x, cos, sin, past=None, key_mask=None):
        a, present = self.attn(self.attn_norm(x), cos, sin, past, key_mask)
        x = x + a
        if self.use_moe:
            f, aux = self.ffn(self.ffn_norm(x))
        else:
            f, aux = self.ffn(self.ffn_norm(x)), x.new_zeros((), dtype=torch.float32)
        return x + f, present, aux


class MTPHead(nn.Module):
    """Predicts token t+2 from (trunk hidden state at t, embedding of token t+1)."""

    def __init__(self, cfg):
        super().__init__()
        self.norm_h = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.norm_e = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.proj = nn.Linear(2 * cfg.d_model, cfg.d_model, bias=False)
        self.attn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.attn = MLA(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.ffn = SwiGLU(cfg.d_model, cfg.mtp_ff)

    def forward(self, h, next_emb, cos, sin):
        z = self.proj(torch.cat([self.norm_h(h), self.norm_e(next_emb)], dim=-1))
        a, _ = self.attn(self.attn_norm(z), cos, sin)
        z = z + a
        return z + self.ffn(self.ffn_norm(z))


class MoEGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks = nn.ModuleList([Block(cfg, use_moe=i >= cfg.n_dense_layers) for i in range(cfg.n_layer)])
        self.final_norm = RMSNorm(cfg.d_model, cfg.norm_eps)
        self.mtp_head = MTPHead(cfg) if cfg.mtp else None

        cos, sin = build_rope_cache(cfg.qk_rope, cfg.max_seq_len, cfg.rope_base)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        scale = 0.02 / math.sqrt(2 * cfg.n_layer)                       # GPT-2 style residual-branch scaling
        for blk in self.blocks:
            nn.init.normal_(blk.attn.out.weight, std=scale)
            if blk.use_moe:
                nn.init.normal_(blk.ffn.w_down, std=scale)
                nn.init.normal_(blk.ffn.shared.down.weight, std=scale)
            else:
                nn.init.normal_(blk.ffn.down.weight, std=scale)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)

    # ---- bookkeeping -------------------------------------------------------
    def num_params(self):
        return sum(p.numel() for p in self.parameters())

    def active_params(self):
        """Parameters touched per token: everything except the routed experts that were not picked."""
        total = self.num_params()
        routed = sum(b.ffn.w_gate_up.numel() + b.ffn.w_down.numel() for b in self.blocks if b.use_moe)
        inactive = routed * (1 - self.cfg.top_k / self.cfg.n_routed)
        mtp = sum(p.numel() for p in self.mtp_head.parameters()) if self.mtp_head is not None else 0
        return int(total - inactive - mtp)                              # MTP is training-only

    @torch.no_grad()
    def update_balance(self, speed):
        """
        Aux-loss-free balancing step (call once per optimizer step on every rank):
        push each expert's routing bias toward the mean load, then reset the counters.
        Returns (max load violation, fraction of assignments dropped) for logging.
        """
        moes = [b.ffn for b in self.blocks if b.use_moe]
        load = torch.stack([m.load for m in moes])
        dropped = torch.stack([m.dropped for m in moes])
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(load)
            dist.all_reduce(dropped)
        mean = load.mean(dim=-1, keepdim=True)
        if speed > 0:
            step = speed * torch.sign(mean - load)                      # under-used experts get a boost
            for m, s in zip(moes, step):
                m.bias += s
        maxvio = ((load - mean).abs().amax(dim=-1) / mean.squeeze(-1).clamp(min=1)).mean()
        drop_frac = dropped.sum() / load.sum().clamp(min=1)
        for m in moes:
            m.load.zero_()
            m.dropped.zero_()
        return maxvio, drop_frac

    def hidden_states(self, idx):
        """Final-layer hidden states (B, T, d_model) - what a Phase 2-7 task head sits on top of."""
        B, T = idx.shape
        cos = self.rope_cos[:T]
        sin = self.rope_sin[:T]
        x = self.tok_emb(idx)
        for blk in self.blocks:
            x, _, _ = blk(x, cos, sin)
        return self.final_norm(x)

    # ---- forward -----------------------------------------------------------
    def forward(self, idx, targets=None, caches=None, start_pos=0, key_mask=None, last_only=False):
        """
        Training:   model(idx, targets) -> (ce, mtp_ce, seq_aux, z_loss)
        Inference:  model(idx, caches=..., ...) -> (logits, new_caches)
        """
        B, T = idx.shape
        cos = self.rope_cos[start_pos:start_pos + T]
        sin = self.rope_sin[start_pos:start_pos + T]

        x = self.tok_emb(idx)
        new_caches, aux_total = [], x.new_zeros((), dtype=torch.float32)
        for i, blk in enumerate(self.blocks):
            past = caches[i] if caches is not None else None
            x, present, aux = blk(x, cos, sin, past, key_mask)
            new_caches.append(present)
            aux_total = aux_total + aux

        hidden = x
        x = self.final_norm(x)
        if last_only:
            x = x[:, -1:, :]
        logits = F.linear(x, self.tok_emb.weight)

        if targets is None:
            return logits, new_caches

        V = logits.size(-1)
        logits = logits.float()
        ce = F.cross_entropy(logits.view(-1, V), targets.reshape(-1))
        z = torch.logsumexp(logits, dim=-1).pow(2).mean()

        mtp_ce = ce.new_zeros(())
        if self.mtp_head is not None and self.training:
            z2 = self.mtp_head(hidden, self.tok_emb(targets), cos, sin)         # embedding of token t+1
            lg2 = F.linear(self.final_norm(z2), self.tok_emb.weight).float()
            shifted = torch.cat([targets[:, 1:], torch.full_like(targets[:, :1], -100)], dim=1)   # token t+2
            mtp_ce = F.cross_entropy(lg2.view(-1, V), shifted.reshape(-1), ignore_index=-100)
        return ce, mtp_ce, aux_total / max(1, len(self.blocks) - self.cfg.n_dense_layers), z


# ---------------------------------------------------------------------------
# Generation (same batched, left-padded KV-cache sampler as arch 3)
# ---------------------------------------------------------------------------
def _sample(logits, temperature, top_k, top_p):
    logits = logits / max(temperature, 1e-5)
    k = min(top_k, logits.size(-1)) if top_k > 0 else logits.size(-1)
    vals, idx = torch.topk(logits, k, dim=-1)
    probs = F.softmax(vals, dim=-1)
    if top_p < 1.0:
        before = probs.cumsum(-1) - probs
        vals = vals.masked_fill(before >= top_p, float("-inf"))
        probs = F.softmax(vals, dim=-1)
    return idx.gather(-1, torch.multinomial(probs, 1)).squeeze(-1)


@torch.no_grad()
def generate(model, prompts, tokenizer, max_new_tokens=384, target_words=128,
             temperature=0.7, top_k=50, top_p=0.92, check_every=8, suppress_eos=True):
    """prompts: list of token-id lists. Returns the generated continuations as strings."""
    model.eval()
    dev = next(model.parameters()).device
    B = len(prompts)
    L = max(len(p) for p in prompts)
    assert L + max_new_tokens <= model.cfg.max_seq_len, "prompt + continuation exceeds context"
    pad, eos = tokenizer.pad_id, tokenizer.eos_id

    ids = torch.full((B, L), pad, dtype=torch.long)
    valid = torch.zeros((B, L), dtype=torch.bool)
    for r, p in enumerate(prompts):
        ids[r, L - len(p):] = torch.tensor(p, dtype=torch.long)
        valid[r, L - len(p):] = True
    ids, valid = ids.to(dev), valid.to(dev)

    causal = torch.tril(torch.ones(L, L, dtype=torch.bool, device=dev))
    mask = (causal[None, None] & valid[:, None, None, :]) | torch.eye(L, dtype=torch.bool, device=dev)[None, None]
    logits, caches = model(ids, caches=None, start_pos=0, key_mask=mask, last_only=True)

    out = torch.empty((B, 0), dtype=torch.long, device=dev)
    finished = torch.zeros(B, dtype=torch.bool, device=dev)
    for step in range(max_new_tokens):
        step_logits = logits[:, -1].float()
        step_logits[:, pad] = float("-inf")
        if suppress_eos:                          # the task wants a full-length continuation: stop on word count only
            step_logits[:, eos] = float("-inf")
        nxt = _sample(step_logits, temperature, top_k, top_p)
        nxt = torch.where(finished, torch.full_like(nxt, pad), nxt)
        out = torch.cat([out, nxt[:, None]], dim=1)
        finished |= nxt == eos

        if (step + 1) % check_every == 0:
            for r in torch.nonzero(~finished).flatten().tolist():
                if len(tokenizer.decode(out[r].tolist()).split()) >= target_words:
                    finished[r] = True
        if bool(finished.all()):
            break

        valid = torch.cat([valid, torch.ones((B, 1), dtype=torch.bool, device=dev)], dim=1)
        logits, caches = model(nxt[:, None], caches=caches, start_pos=L + step,
                               key_mask=valid[:, None, None, :], last_only=True)

    texts = []
    for r in range(B):
        toks = out[r].tolist()
        if eos in toks:
            toks = toks[:toks.index(eos)]
        toks = [t for t in toks if t != pad]
        texts.append(" ".join(tokenizer.decode(toks).split()[:target_words]))
    return texts

In [ ]:
%%writefile /kaggle/working/phase4_arch4/hub_files.py
"""Finds Phase-1 files: an attached Kaggle dataset first, then a local cache, then Hugging Face."""
import os
import re

from huggingface_hub import hf_hub_download

KAGGLE_INPUT = "/kaggle/input"


def _squash(text):
    return re.sub(r"[^a-z0-9]", "", text.lower())


def find_in_kaggle_input(filename, hint):
    """Only accepts a file whose path mentions the architecture (e.g. 'arch3', 'arch-3', 'Arch_3'),
    because arch 2, 3 and 4 all call their weights model_final.pt."""
    if not os.path.isdir(KAGGLE_INPUT):
        return None
    hits = []
    for root, _, files in os.walk(KAGGLE_INPUT):
        if filename in files:
            path = os.path.join(root, filename)
            if _squash(hint) in _squash(path):
                hits.append(path)
    return sorted(hits)[0] if hits else None


def fetch_from_hub(repo, filename, local_dir):
    try:
        return hf_hub_download(repo, filename, local_dir=local_dir, token=os.environ.get("HF_TOKEN"))
    except Exception:
        return None


def locate(filenames, repos, hint, local_dir):
    """Names are tried in order of preference. Returns (path, where it came from)."""
    for name in filenames:
        path = find_in_kaggle_input(os.path.basename(name), hint)
        if path:
            return path, "kaggle input"
    for name in filenames:
        for repo in repos:
            cached = os.path.join(local_dir, _squash(repo), name)
            if os.path.exists(cached):
                return cached, f"cache of hf://{repo}"
            path = fetch_from_hub(repo, name, os.path.join(local_dir, _squash(repo)))
            if path:
                return path, f"hf://{repo}"
    raise FileNotFoundError(f"None of {filenames} found under /kaggle/input (path must mention '{hint}') or in "
                            f"{repos}. Attach the Phase-1 weights as a Kaggle dataset, or check HF_TOKEN and the repo names.")

In [ ]:
%%writefile /kaggle/working/phase4_arch4/backbone.py
"""Loads the Phase-1 arch 4 weights (DeepSeek-style MoE decoder) and exposes them as a sentence encoder for rhetorical roles."""
import os

import torch
import torch.nn as nn

import config as C
from hub_files import locate, fetch_from_hub  # noqa: F401  (fetch_from_hub is used by run_rr)
from phase1_model import MoEGPT, ModelConfig
from tokenizer import NerTokenizer


def add_prompt(x, mask, prompt):
    """Prompt tuning (Lester et al. 2021): prepend learned vectors to the token embeddings."""
    if prompt is None:
        return x, mask, 0
    P = prompt.size(0)
    x = torch.cat([prompt.to(x.dtype).unsqueeze(0).expand(x.size(0), -1, -1), x], dim=1)
    mask = torch.cat([mask.new_ones(mask.size(0), P), mask], dim=1)
    return x, mask, P


def load_tokenizer():
    path, _ = locate(C.TOKENIZER_FILES, C.TOKENIZER_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    tok = NerTokenizer.load(path)
    tok.sep_id = tok.eos_id               # marks the start of every sentence inside a pack
    return tok


class Backbone(nn.Module):
    def __init__(self, gpt, keep_layers, bidirectional):
        super().__init__()
        self.d_model = gpt.cfg.d_model
        self.tok_emb = gpt.tok_emb
        blocks = gpt.blocks if keep_layers is None else [gpt.blocks[i] for i in keep_layers]
        self.layers = nn.ModuleList(blocks)
        self.final_norm = gpt.final_norm
        self.register_buffer("rope_cos", gpt.rope_cos.clone(), persistent=False)
        self.register_buffer("rope_sin", gpt.rope_sin.clone(), persistent=False)
        self.bidirectional = bidirectional
        for blk in self.layers:
            blk.attn.causal = not bidirectional
            if blk.use_moe:
                blk.ffn.dropless = True        # never drop tokens while fine-tuning; routing biases stay frozen

    def forward(self, ids, mask, prompt=None):
        x, mask, P = add_prompt(self.tok_emb(ids), mask, prompt)
        T = x.size(1)
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        key_mask = mask[:, None, None, :] if (self.bidirectional and not bool(mask.all())) else None
        for blk in self.layers:
            x, _, _ = blk(x, cos, sin, None, key_mask)
        return self.final_norm(x)[:, P:]


def build_backbone(keep_layers=None, bidirectional=True):
    path, source = locate(C.WEIGHT_FILES, C.PHASE1_REPOS, C.KAGGLE_HINT, C.CACHE_DIR)
    blob = torch.load(path, map_location="cpu")
    cfg = dict(blob["model_cfg"], mtp=False)               # the multi-token head is a pre-training-only extra
    gpt = MoEGPT(ModelConfig(**cfg))
    state = {k: v.float() for k, v in blob["model"].items() if not k.startswith("mtp_head.")}
    gpt.load_state_dict(state)
    if os.environ.get("P2_VERBOSE", "1") == "1":
        print(f"Phase-1 arch 4 weights: {path} ({source})", flush=True)
    return Backbone(gpt, keep_layers, bidirectional)


def student_layers():
    return list(C.STUDENT_LAYERS)

In [ ]:
%%writefile /kaggle/working/phase4_arch4/cjpe_core.py
"""
Phase 4 (court judgment prediction) machinery shared by all four architectures.

Every case is turned into a few 512-token chunks: the opening of the judgment plus its last part (ILDC, Malik et al.
2021, found the closing section - right before the removed verdict - to be the most predictive). The Phase-1 backbone
encodes each chunk, chunks are mean-pooled, tagged with where they came from, and combined by learned attention pooling
into one document vector (a hierarchical chunk -> document representation) that a linear layer classifies.
"""
import os
import json
import math
from multiprocessing import Pool

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F


# ------------------------------------------------------------------------------------------- data
def load_cases(path, with_labels):
    """Only id, text and (for train/dev) label are read. The public test file also carries a 'label' field;
    it is deliberately ignored."""
    with open(path, encoding="utf-8") as f:
        rows = json.load(f)
    ids = [r["id"] for r in rows]
    texts = [r["text"] for r in rows]
    labels = [int(r["label"]) for r in rows] if with_labels else None
    return ids, texts, labels


_tok = None
_cfg = None


def _init_worker(cfg_dict):
    global _tok, _cfg
    import backbone as BB
    _tok = BB.load_tokenizer()
    _cfg = cfg_dict


def chunk_text(text):
    """-> (K, W) token ids and per-chunk lengths. Short cases are simply cut into consecutive chunks."""
    W, head, tail = _cfg["W"], _cfg["head"], _cfg["tail"]
    K = head + tail
    budget = K * W
    approx = budget * _cfg["chars_per_token"]
    if len(text) <= approx:
        ids = _tok.encode(text)
    else:
        ids = None
    if ids is not None and len(ids) <= budget:
        pieces = [ids[i:i + W] for i in range(0, len(ids), W)]
    else:
        h = _tok.encode(text[:head * W * _cfg["chars_per_token"]])[:head * W]
        t = _tok.encode(text[-tail * W * _cfg["chars_per_token"]:])[-tail * W:]
        pieces = [h[i:i + W] for i in range(0, len(h), W)][:head] + [t[i:i + W] for i in range(0, len(t), W)][-tail:]
    out = np.full((K, W), _cfg["pad"], dtype=np.int32)
    lens = np.zeros(K, dtype=np.int16)
    for k, p in enumerate(pieces[:K]):
        out[k, :len(p)] = p
        lens[k] = len(p)
    return out, lens


def prepare_split(texts, cfg_dict, workers):
    with Pool(workers, initializer=_init_worker, initargs=(cfg_dict,)) as pool:
        res = pool.map(chunk_text, texts, chunksize=64)
    return np.stack([r[0] for r in res]), np.stack([r[1] for r in res])


# ----------------------------------------------------------------------------------- adaptation
class LoRMALinear(nn.Module):
    """Bihany, Patel & Modi 2025 (LoRMA+): h = (alpha/r B A + I) W0 x, computed as W0x -> A -> B."""

    def __init__(self, base, r, alpha, dropout):
        super().__init__()
        self.base = base
        self.scale = alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = F.linear(x, self.base.weight)
        h = h + F.linear(F.linear(self.drop(h), self.A), self.B) * self.scale
        return h + self.base.bias if self.base.bias is not None else h


class BitFitLinear(nn.Module):
    """BitFit (Ben Zaken et al. 2022) trains only bias terms. Arch 2-4 were pre-trained without biases, so a
    zero-initialised bias is added there (the model is unchanged at the start); arch 1 keeps its own biases."""

    def __init__(self, base):
        super().__init__()
        self.base = base
        self.bias = nn.Parameter(torch.zeros(base.out_features)) if base.bias is None else None

    def forward(self, x):
        out = self.base(x)
        return out + self.bias if self.bias is not None else out


class Adapter(nn.Module):
    """Houlsby et al. 2019 bottleneck adapter on a sub-layer's output: h + up(gelu(down(h))). up = 0 at the start."""

    def __init__(self, base, bottleneck):
        super().__init__()
        self.base = base
        d = base.out_features
        self.down = nn.Linear(d, bottleneck)
        self.up = nn.Linear(bottleneck, d)
        nn.init.normal_(self.down.weight, std=0.02)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        h = self.base(x)
        return h + self.up(F.gelu(self.down(h)))


def wrap_linears(root, names, make, skip=("router",)):
    """Wrap matching nn.Linear children. MoE routers are always left alone: arch 4 reads router.weight directly,
    and the pre-trained routing should not drift."""
    n = 0
    for parent in list(root.modules()):
        for name, child in list(parent.named_children()):
            if name in skip:
                continue
            if (names is None or name in names) and isinstance(child, nn.Linear):
                setattr(parent, name, make(child))
                n += 1
    return n


# ---------------------------------------------------------------------------------------- model
class CJPEModel(nn.Module):
    def __init__(self, backbone, cfg):
        super().__init__()
        self.backbone = backbone
        D = backbone.d_model
        K = cfg.HEAD_CHUNKS + cfg.TAIL_CHUNKS
        self.chunk_pos = nn.Embedding(K, D)                 # which part of the judgment a chunk came from
        nn.init.zeros_(self.chunk_pos.weight)
        self.att = nn.Sequential(nn.Linear(D, cfg.POOL_DIM), nn.Tanh(), nn.Linear(cfg.POOL_DIM, 1))
        self.drop = nn.Dropout(cfg.HEAD_DROPOUT)
        self.cls = nn.Linear(D, 2)

    def encode_chunks(self, ids, lens):
        """ids (B, K, W), lens (B, K) -> chunk vectors (B, K, D); empty chunks are never run through the backbone."""
        B, K, W = ids.shape
        flat_ids, flat_lens = ids.view(B * K, W), lens.view(B * K)
        real = flat_lens > 0
        vecs = torch.zeros(B * K, self.backbone.d_model, device=ids.device)
        if real.any():
            sub = flat_ids[real]
            L = int(flat_lens[real].max())
            sub = sub[:, :L]
            mask = torch.arange(L, device=ids.device)[None, :] < flat_lens[real][:, None]
            h = self.backbone(sub, mask).float()
            pooled = (h * mask[..., None]).sum(1) / mask.sum(1, keepdim=True).clamp(min=1)
            vecs = vecs.index_put((real.nonzero().squeeze(1),), pooled)
        return vecs.view(B, K, -1), (lens > 0)

    def forward(self, ids, lens):
        chunks, present = self.encode_chunks(ids, lens)
        chunks = chunks + self.chunk_pos.weight[None]
        scores = self.att(chunks).squeeze(-1).masked_fill(~present, float("-inf"))
        weights = torch.softmax(scores, dim=-1)
        doc = (weights[..., None] * chunks).sum(1)
        return self.cls(self.drop(doc)).float(), doc, weights


def configure_method(model, method, cfg):
    bb = model.backbone
    if method in ("full", "kd"):
        for p in model.parameters():
            p.requires_grad = True
        return "all weights trainable"
    for p in bb.parameters():
        p.requires_grad = False
    if method == "lorma":
        n = wrap_linears(bb.layers, cfg.ADAPTER_TARGETS, lambda m: LoRMALinear(m, cfg.ADAPTER_RANK, cfg.LORMA_ALPHA, cfg.ADAPTER_DROPOUT))
        return f"LoRMA on {n} attention projections (r={cfg.ADAPTER_RANK})"
    if method == "bitfit":
        n = wrap_linears(bb.layers, None, BitFitLinear)
        for name, p in bb.named_parameters():
            if name.endswith(".bias"):
                p.requires_grad = True
        return f"bias terms of {n} linear layers trainable"
    if method == "adapter":
        n = wrap_linears(bb.layers, cfg.ADAPTER_SITES, lambda m: Adapter(m, cfg.ADAPTER_BOTTLENECK))
        return f"{n} bottleneck adapters (size {cfg.ADAPTER_BOTTLENECK}) after attention and feed-forward outputs"
    if method == "unfreeze":
        return "gradual unfreezing: head first, then one more backbone layer at a time from the top"
    raise ValueError(method)


def set_unfrozen(model, n_top):
    """Gradual unfreezing (Howard & Ruder 2018): only the top n_top layers (+ final norm) of the backbone train."""
    layers = list(model.backbone.layers)
    for i, layer in enumerate(layers):
        for p in layer.parameters():
            p.requires_grad = i >= len(layers) - n_top
    for p in model.backbone.final_norm.parameters():
        p.requires_grad = n_top > 0


def count_params(model):
    return sum(p.numel() for p in model.parameters()), sum(p.numel() for p in model.parameters() if p.requires_grad)


def trainable_state(model, method):
    if method == "unfreeze":                                # every layer may have moved by the end
        names = {n for n, _ in model.named_parameters()}
    else:
        names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().float().half() for k, v in model.state_dict().items() if k in names}


def layer_depth(name, n_layers):
    if ".layers." in name:
        return int(name.split(".layers.")[1].split(".")[0]) + 1
    if "emb" in name and name.startswith("backbone."):
        return 0
    return n_layers + 1


def build_optimizer(model, method, cfg):
    n_layers = len(model.backbone.layers)
    groups = {}
    for name, p in model.named_parameters():
        if method != "unfreeze" and not p.requires_grad:
            continue
        if not name.startswith("backbone."):
            lr = cfg.HEAD_LR
        elif method in ("full", "kd", "unfreeze"):
            lr = cfg.FULL_LR * cfg.LLRD ** (n_layers + 1 - layer_depth(name, n_layers))
        elif method == "bitfit":
            lr = cfg.BITFIT_LR
        else:
            lr = cfg.ADAPTER_LR
        wd = 0.0 if p.ndim < 2 else cfg.WEIGHT_DECAY
        groups.setdefault((lr, wd), []).append(p)
    return torch.optim.AdamW([{"params": ps, "lr": lr, "weight_decay": wd} for (lr, wd), ps in groups.items()],
                             betas=(0.9, 0.98), eps=1e-6)


# --------------------------------------------------------------------------------------- metric
def binary_scores(gold, pred):
    gold, pred = np.asarray(gold), np.asarray(pred)
    per = {}
    for c in (0, 1):
        tp = int(((gold == c) & (pred == c)).sum())
        p = tp / max(1, int((pred == c).sum()))
        r = tp / max(1, int((gold == c).sum()))
        per[c] = (p, r, 2 * p * r / (p + r) if p + r else 0.0)
    return {"f1": (per[0][2] + per[1][2]) / 2, "p": (per[0][0] + per[1][0]) / 2, "r": (per[0][1] + per[1][1]) / 2,
            "acc": float((gold == pred).mean())}


def best_threshold(gold, prob1):
    """Decision threshold on P(class 1) that maximises dev macro-F1 (train is 59/41, dev and test are balanced)."""
    best = (0.5, binary_scores(gold, (prob1 >= 0.5).astype(int))["f1"])
    candidates = np.unique(np.concatenate([np.linspace(0.05, 0.95, 91), np.quantile(prob1, np.linspace(0.02, 0.98, 97))]))
    for t in candidates:
        f1 = binary_scores(gold, (prob1 >= t).astype(int))["f1"]
        if f1 > best[1]:
            best = (float(t), f1)
    return best


@torch.no_grad()
def predict_probs(model, ids, lens, device, batch):
    model.eval()
    out = []
    for b in range(0, len(ids), batch):
        x = torch.from_numpy(ids[b:b + batch].astype(np.int64)).to(device)
        l = torch.from_numpy(lens[b:b + batch].astype(np.int64)).to(device)
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            logits, _, _ = model(x, l)
        out.append(torch.softmax(logits.float(), -1)[:, 1].cpu().numpy())
    return np.concatenate(out)


def write_submission(test_ids, preds, out_dir, zip_name):
    import zipfile
    assert len(test_ids) == len(preds) == len(set(test_ids))
    payload = [{"id": i, "pred": int(p)} for i, p in zip(test_ids, preds)]
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w") as f:
        json.dump(payload, f)
    zip_path = os.path.join(out_dir, zip_name)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zip_path) as z:
        assert z.namelist() == ["prediction.json"]
    return path, zip_path

In [ ]:
%%writefile /kaggle/working/phase4_arch4/run_cjpe.py
"""
One Phase-4 experiment = one call:

    python run_cjpe.py --method lorma

Trains for a fixed wall-clock budget (37K long cases are far more than one Kaggle session can read in full), checks a
dev subset along the way and keeps the best checkpoint, tunes the decision threshold on the whole dev set, then writes
the test predictions + submission zip and pushes everything to Hugging Face.
"""
import os
import sys
import json
import math
import time
import random
import argparse

import numpy as np
import torch
import torch.nn.functional as F

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import config as C
import backbone as BB
from cjpe_core import (CJPEModel, configure_method, set_unfrozen, count_params, trainable_state, build_optimizer,
                       binary_scores, best_threshold, predict_probs, write_submission)


def log(*a):
    print(*a, flush=True)


def build_model(method, keep_layers=None):
    model = CJPEModel(BB.build_backbone(keep_layers=keep_layers, bidirectional=C.BIDIRECTIONAL), C)
    note = configure_method(model, method, C)
    return model, note


def load_finetuned(method, device):
    path = os.path.join(C.OUT_DIR, method, f"cjpe_{method}.pt")
    if not os.path.exists(path):
        path = BB.fetch_from_hub(C.CJPE_REPO, f"{method}/cjpe_{method}.pt", os.path.join(C.OUT_DIR, "_hub"))
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(f"No fine-tuned '{method}' model locally or on {C.CJPE_REPO}; run '{method}' first.")
    blob = torch.load(path, map_location="cpu")
    model, _ = build_model(method, blob.get("keep_layers"))
    _, unexpected = model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    assert not unexpected, unexpected
    return model.to(device).eval()


def push(files, message):
    if not os.environ.get("HF_TOKEN"):
        return
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(C.CJPE_REPO, private=True, exist_ok=True, token=os.environ["HF_TOKEN"])
        for local, remote in files:
            api.upload_file(path_or_fileobj=local, path_in_repo=remote, repo_id=C.CJPE_REPO,
                            token=os.environ["HF_TOKEN"], commit_message=message)
        log(f"pushed to {C.CJPE_REPO}: {message}")
    except Exception as e:
        log(f"HF push failed ({type(e).__name__}: {e}) - the files are still in {C.OUT_DIR}")


def load_npz(name):
    z = np.load(os.path.join(C.DATA_DIR, f"{name}.npz"))
    return z["ids"], z["lens"], (z["labels"] if "labels" in z.files else None)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", required=True, choices=["full", "lorma", "bitfit", "adapter", "unfreeze", "kd"])
    method = ap.parse_args().method
    random.seed(C.SEED)
    np.random.seed(C.SEED)
    torch.manual_seed(C.SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    out_dir = os.path.join(C.OUT_DIR, method)
    os.makedirs(out_dir, exist_ok=True)
    t0 = time.time()

    tr_ids, tr_lens, tr_y = load_npz("train")
    dv_ids, dv_lens, dv_y = load_npz("dev")
    te_ids, te_lens, _ = load_npz("test")
    with open(os.path.join(C.DATA_DIR, "test_ids.json")) as f:
        test_case_ids = json.load(f)
    if C.TRAIN_DOC_LIMIT:
        tr_ids, tr_lens, tr_y = tr_ids[:C.TRAIN_DOC_LIMIT], tr_lens[:C.TRAIN_DOC_LIMIT], tr_y[:C.TRAIN_DOC_LIMIT]
    sub = np.random.RandomState(0).permutation(len(dv_y))[:C.DEV_SUBSET]          # quick check during training
    if C.DEV_FINAL_LIMIT:
        dv_ids, dv_lens, dv_y = dv_ids[:C.DEV_FINAL_LIMIT], dv_lens[:C.DEV_FINAL_LIMIT], dv_y[:C.DEV_FINAL_LIMIT]
        sub = sub[sub < len(dv_y)]
    log(f"[{method}] train {len(tr_y):,} cases (class 1: {tr_y.mean():.0%}) | dev {len(dv_y):,} | test {len(test_case_ids):,} "
        f"| {tr_ids.shape[1]} chunks x {tr_ids.shape[2]} tokens per case")

    keep = BB.student_layers() if method == "kd" else None
    model, note = build_model(method, keep)
    model.to(device)
    if method == "unfreeze":
        set_unfrozen(model, 0)
    total, trainable = count_params(model)
    log(f"[{method}] {note} | params {total / 1e6:.1f}M, trainable now {trainable / 1e6:.2f}M | budget {C.TRAIN_MINUTES} min")

    teacher = None
    if method == "kd":
        teacher = load_finetuned(C.KD_TEACHER, device)
        log(f"[kd] teacher = fine-tuned '{C.KD_TEACHER}' ({count_params(teacher)[0] / 1e6:.1f}M) -> "
            f"student with backbone layers {keep} ({total / 1e6:.1f}M)")

    counts = np.bincount(tr_y, minlength=2)
    class_w = torch.tensor(len(tr_y) / (2 * counts), dtype=torch.float32, device=device)   # balanced loss

    optimizer = build_optimizer(model, method, C)
    base_lrs = [g["lr"] for g in optimizer.param_groups]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    docs_per_step = C.MICRO_DOCS * C.GRAD_ACCUM
    step_cap = max(1, C.MAX_EPOCHS * len(tr_y) // docs_per_step)
    budget = C.TRAIN_MINUTES * 60
    n_layers = len(model.backbone.layers)

    order = np.random.RandomState(C.SEED).permutation(len(tr_y))
    cursor, step, seen, running = 0, 0, 0, 0.0
    best, history = {"f1": -1.0}, []
    t_train = time.time()

    def progress():
        return max(step / step_cap, (time.time() - t_train) / budget)

    while progress() < 1.0:
        p = progress()
        if method == "unfreeze":
            set_unfrozen(model, min(n_layers, 1 + int(p / C.UNFREEZE_BY * n_layers)))   # all layers live by UNFREEZE_BY
        warm = min(1.0, (step + 1) / C.WARMUP_STEPS)
        lr_scale = warm * (C.MIN_LR_RATIO + (1 - C.MIN_LR_RATIO) * 0.5 * (1 + math.cos(math.pi * min(1.0, p))))
        for g, lr in zip(optimizer.param_groups, base_lrs):
            g["lr"] = lr * lr_scale

        model.train()
        for _ in range(C.GRAD_ACCUM):
            if cursor + C.MICRO_DOCS > len(order):
                order = np.random.RandomState(C.SEED + step).permutation(len(tr_y))
                cursor = 0
            idx = np.sort(order[cursor:cursor + C.MICRO_DOCS])
            cursor += C.MICRO_DOCS
            x = torch.from_numpy(tr_ids[idx].astype(np.int64)).to(device)
            l = torch.from_numpy(tr_lens[idx].astype(np.int64)).to(device)
            y = torch.from_numpy(tr_y[idx].astype(np.int64)).to(device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                logits, doc, _ = model(x, l)
                loss = F.cross_entropy(logits, y, weight=class_w)
                if teacher is not None:
                    with torch.no_grad():
                        t_logits, t_doc, _ = teacher(x, l)
                    T = C.KD_TEMPERATURE
                    kl = F.kl_div(F.log_softmax(logits / T, -1), F.softmax(t_logits.float() / T, -1), reduction="batchmean") * T * T
                    rep = (1 - F.cosine_similarity(doc.float(), t_doc.float(), dim=-1)).mean()
                    loss = C.KD_ALPHA * loss + (1 - C.KD_ALPHA) * kl + C.KD_HIDDEN_WEIGHT * rep
            scaler.scale(loss / C.GRAD_ACCUM).backward()
            running += loss.item()
            seen += 1
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_([q for q in model.parameters() if q.requires_grad], 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        step += 1

        if step % C.EVAL_EVERY_STEPS == 0 or progress() >= 1.0:
            probs = predict_probs(model, dv_ids[sub], dv_lens[sub], device, C.EVAL_BATCH)
            thr, f1 = best_threshold(dv_y[sub], probs)
            history.append({"step": step, "docs_seen": step * docs_per_step, "train_loss": running / max(1, seen),
                            "dev_subset_f1": f1, "minutes": (time.time() - t0) / 60})
            log(f"[{method}] step {step} | {step * docs_per_step:,} cases seen | loss {running / max(1, seen):.4f} | "
                f"dev-subset macro-F1 {f1:.4f} (thr {thr:.2f}) | {100 * min(1.0, progress()):.0f}% of budget")
            running, seen = 0.0, 0
            if f1 > best["f1"]:
                best = {"f1": f1, "step": step}
                torch.save({"method": method, "keep_layers": keep, "weights": trainable_state(model, method),
                            "step": step}, os.path.join(out_dir, f"cjpe_{method}.pt"))

    # ------------------------------------- best checkpoint: threshold on full dev, then test
    blob = torch.load(os.path.join(out_dir, f"cjpe_{method}.pt"), map_location="cpu")
    model.load_state_dict({k: v.float() for k, v in blob["weights"].items()}, strict=False)
    t_inf = time.time()
    dev_probs = predict_probs(model, dv_ids, dv_lens, device, C.EVAL_BATCH)
    dev_seconds = time.time() - t_inf
    thr, _ = best_threshold(dv_y, dev_probs)
    dev_pred = (dev_probs >= thr).astype(int)
    dev = binary_scores(dv_y, dev_pred)
    dev_at_half = binary_scores(dv_y, (dev_probs >= 0.5).astype(int))
    fp = int(((dev_pred == 1) & (dv_y == 0)).sum())
    fn = int(((dev_pred == 0) & (dv_y == 1)).sum())

    test_probs = predict_probs(model, te_ids, te_lens, device, C.EVAL_BATCH)
    test_pred = (test_probs >= thr).astype(int)
    _, zip_path = write_submission(test_case_ids, test_pred, out_dir, f"submission_{C.ARCH}_{method}.zip")
    np.save(os.path.join(out_dir, "dev_probs.npy"), dev_probs)

    summary = {
        "arch": C.ARCH, "method": method, "best_step": best["step"], "threshold": thr,
        "dev_macro_f1": dev["f1"], "dev_macro_p": dev["p"], "dev_macro_r": dev["r"], "dev_acc": dev["acc"],
        "dev_macro_f1_at_0.5": dev_at_half["f1"], "dev_false_pos": fp, "dev_false_neg": fn,
        "test_pred_class1_share": float(test_pred.mean()),
        "params_total": total, "params_trainable": count_params(model)[1] if method != "unfreeze" else total,
        "dev_inference_seconds": round(dev_seconds, 1),
        "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2) if device.type == "cuda" else 0.0,
        "minutes": round((time.time() - t0) / 60, 1), "cases_seen": step * docs_per_step, "history": history,
    }
    with open(os.path.join(out_dir, "results.json"), "w") as f:
        json.dump(summary, f, indent=2)
    log(f"[{method}] DONE | dev macro-F1 {dev['f1']:.4f} (threshold {thr:.2f}; {dev_at_half['f1']:.4f} at 0.5) | "
        f"FP {fp} FN {fn} | test predicted class 1: {test_pred.mean():.0%} -> {zip_path}")

    push([(os.path.join(out_dir, f"cjpe_{method}.pt"), f"{method}/cjpe_{method}.pt"),
          (os.path.join(out_dir, "results.json"), f"{method}/results.json"),
          (zip_path, f"{method}/{os.path.basename(zip_path)}")], f"{C.ARCH} {method}: dev macro-F1 {dev['f1']:.4f}")


if __name__ == "__main__":
    main()

## 4. Data: download and turn every case into chunks
The cases are tokenized once with all CPU cores and stored as compact arrays, so each experiment starts instantly.

In [ ]:
import importlib
import config as C
importlib.reload(C)
import backbone as BB
import cjpe_core as K
importlib.reload(K)

for split in ("train", "dev", "test"):
    folder = "Test" if split == "test" else "Train"
    hf_hub_download(C.HF_DATASET_REPO, f"Phase_4_CJPE/{folder}/{split}.json", repo_type="dataset",
                    local_dir=C.DATA_DIR, token=HF_TOKEN)

tok = BB.load_tokenizer()                       # caches the tokenizer before the worker processes start
chunk_cfg = {"W": C.CHUNK_TOKENS, "head": C.HEAD_CHUNKS, "tail": C.TAIL_CHUNKS,
             "chars_per_token": C.CHARS_PER_TOKEN, "pad": tok.pad_id}

for split in ("train", "dev", "test"):
    t0 = time.time()
    ids, texts, labels = K.load_cases(C.RAW[split], with_labels=split != "test")
    chunk_ids, chunk_lens = K.prepare_split(texts, chunk_cfg, C.PREP_WORKERS)
    arrays = {"ids": chunk_ids, "lens": chunk_lens}
    if labels is not None:
        arrays["labels"] = np.asarray(labels, dtype=np.int64)
    np.savez(os.path.join(C.DATA_DIR, f"{split}.npz"), **arrays)
    if split == "test":
        json.dump(ids, open(os.path.join(C.DATA_DIR, "test_ids.json"), "w"))
    used = (chunk_lens > 0).sum(1)
    extra = f" | class 1: {np.mean(labels):.0%}" if labels is not None else ""
    print(f"{split}: {len(ids):,} cases, chunks used per case {used.mean():.2f} (max {chunk_lens.shape[1]}) "
          f"| {time.time() - t0:.0f}s{extra}")
    del texts, chunk_ids, chunk_lens, arrays
    gc.collect()

## 5. Load the Phase-1 weights once
This checks the model end to end and caches the files for the experiment processes.

In [ ]:
probe = K.CJPEModel(BB.build_backbone(), C)
dev = np.load(os.path.join(C.DATA_DIR, "dev.npz"))
with torch.no_grad():
    logits, doc, w = probe(torch.from_numpy(dev["ids"][:2].astype(np.int64)), torch.from_numpy(dev["lens"][:2].astype(np.int64)))
print(f"model ready: {sum(p.numel() for p in probe.parameters()) / 1e6:.1f}M parameters | logits {tuple(logits.shape)} | "
      f"chunk attention of case 0: {np.round(w[0].numpy(), 2)}")
del probe, dev

## 6. Run the experiments
Each experiment is its own process (`run_cjpe.py --method ...`); with two GPUs, two run at once, so the six take about
three rounds of ~`TRAIN_MINUTES` + 10 minutes. `kd` starts as soon as `full` (its teacher) is done.
Results, checkpoints and zips are pushed to `D-240357/legal-cjpe-arch4` as they finish.

In [ ]:
import subprocess, threading

pending = list(C.METHODS)
done, failed = set(), set()
lock = threading.Lock()


def next_job():
    with lock:
        if "kd" in pending and C.KD_TEACHER in failed:
            pending.remove("kd")
            print(f"skipping kd: its teacher '{C.KD_TEACHER}' failed")
        for m in pending:
            if m != "kd" or C.KD_TEACHER in done or C.KD_TEACHER not in C.METHODS:
                pending.remove(m)
                return m
        return "wait" if pending else None


def worker(gpu):
    while True:
        job = next_job()
        if job is None:
            return
        if job == "wait":
            time.sleep(15)
            continue
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        proc = subprocess.Popen([sys.executable, "run_cjpe.py", "--method", job], cwd=WORK_DIR, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(f"[gpu{gpu}] {line}", end="")
        with lock:
            (done if proc.wait() == 0 else failed).add(job)


t0 = time.time()
threads = [threading.Thread(target=worker, args=(g,)) for g in range(max(1, torch.cuda.device_count()))]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(f"\nfinished in {(time.time() - t0) / 60:.0f} min | ok: {sorted(done)} | failed: {sorted(failed)}")

## 7. Results
Dev macro-F1 (threshold tuned on dev), what the threshold bought, error types and the cost of each method.

In [ ]:
import pandas as pd

rows = []
for path in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "results.json"))):
    r = json.load(open(path))
    rows.append({"method": r["method"], "dev macro-F1": r["dev_macro_f1"], "at 0.5": r["dev_macro_f1_at_0.5"],
                 "threshold": r["threshold"], "macro P": r["dev_macro_p"], "macro R": r["dev_macro_r"],
                 "false pos": r["dev_false_pos"], "false neg": r["dev_false_neg"],
                 "cases seen": r["cases_seen"], "params (M)": r["params_total"] / 1e6,
                 "trainable %": 100 * r["params_trainable"] / r["params_total"], "minutes": r["minutes"],
                 "peak GPU GB": r["peak_gpu_gb"], "dev infer s": r["dev_inference_seconds"]})
table = pd.DataFrame(rows).sort_values("dev macro-F1", ascending=False).round(4)
display(table)

## 8. Pick a submission
Every experiment wrote `runs/<method>/submission_<arch>_<method>.zip`: a single `prediction.json` at the root holding a
JSON list of `{"id": ..., "pred": 0 or 1}` for all 1,517 test cases. The best on dev is also copied to
`/kaggle/working/submission.zip`. Upload the zips exactly as they are.

In [ ]:
best = table.iloc[0]["method"]
shutil.copy(glob.glob(os.path.join(C.OUT_DIR, best, "submission_*.zip"))[0], "/kaggle/working/submission.zip")
print(f"best on dev: {best} -> /kaggle/working/submission.zip")
for z in sorted(glob.glob(os.path.join(C.OUT_DIR, "*", "submission_*.zip"))):
    print("  ", z)